<a href="https://colab.research.google.com/github/arivneuraxprojects/AI-RESUME-ANALYZER/blob/main/pdf_question_answeringipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 1: Install Libraries
!pip install -q pypdf transformers sentence-transformers faiss-cpu accelerate langchain langchain-text-splitters pypdf

In [ ]:
# ============================================
# STEP 2: Import Libraries
# ============================================

from google.colab import files
from pypdf import PdfReader
from transformers.utils import logging
logging.set_verbosity_error()
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
#import ipywidgets as widgets
#from IPython.display import display, clear_output
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
# ============================================
# STEP 3: Upload PDF
# ============================================

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]


Saving Introduction-to-AI-and-Basic-Concepts.pdf to Introduction-to-AI-and-Basic-Concepts (3).pdf


In [ ]:
# ============================================
# STEP 4: Extract Text from PDF
# ============================================

text = ""

reader = PdfReader(pdf_file)

for page in reader.pages:
    text += page.extract_text()

print("✅ PDF Loaded Successfully")
print("✅ PDF Text Extracted")

print("\n📄 First 1000 Characters:\n")
print(text[:1000])

✅ PDF Loaded Successfully
✅ PDF Text Extracted

📄 First 1000 Characters:

Introduction to Artificial Intelligence and 
Fundamental Concepts
Dr. Melike PALSÜ KURTWhat Will We Learn?
• What is Artificial Intelligence?
• Can Machines Think, and How Can They Think?
• Turing Test
• Chinese Room
• The AI Lifecycle
• What Can Artificial Intelligence Do?
• What Can Artificial Intelligence Not Do?
• Fundamental Concepts
• Types of Data
• Data Preprocessing
• Feature Engineering
• Good Practice ExamplesWhat is Artificial Intelligence?
What is Artificial Intelligence?
In its most general form, artificial intelligence 
(AI) is defined as the ability of a computer or 
a robot controlled by a computer to perform 
various tasks in a manner similar to 
intelligent beings.
National Artificial Intelligence Strategy of 
Türkiye 2021 - 2025
What is Artificial Intelligence?
Narrow AI is effective 
only in the specific 
areas it has been 
trained for, unlike 
human intelligence; it 
deals with a specific 
s

In [ ]:
# =========================
# STEP 5: Create Chunks
# =========================


chunk_size = 500

chunks = []

for i in range(0, len(text), chunk_size):

    chunk = text[i:i + chunk_size]

    chunks.append(chunk)

print(f"\n✅ Total Chunks: {len(chunks)}")


✅ Total Chunks: 37


In [ ]:
# =========================
# STEP 6: Load Embedding Model
# =========================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding Model Loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding Model Loaded


In [ ]:
# =========================
# STEP 7: Create Embeddings
# =========================

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

embeddings = embeddings.astype("float32")

print("✅ Embeddings Created")

✅ Embeddings Created


In [ ]:
# =========================
# STEP 8: Create FAISS Index
# =========================

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("✅ FAISS Vector Database Ready")

✅ FAISS Vector Database Ready


In [ ]:
# =========================
# STEP 9: Load The Model
# =========================



generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)


print("✅ Load the model")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Load the model


In [ ]:
# =========================
# STEP 10: Chat Loop
# =========================

print("\n🤖 PDF Chatbot Ready!")
print("Type 'exit' to quit.\n")

while True:

    question = input("🧑 You: ")

    if question.lower() == "exit":

        print("\n👋 Chat Ended")
        break

    # =====================================
    # Convert Question to Embedding
    # =====================================

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    question_embedding = question_embedding.astype("float32")

    # =====================================
    # Search Similar Chunks
    # =====================================

    distances, indices = index.search(
        question_embedding,
        k=3
    )

    # =====================================
    # Build Context
    # =====================================

    context = ""

    for idx in indices[0]:

        context += chunks[idx] + "\n"

    # Limit context size for FLAN-T5
    context = context[:1500]

    # =====================================
    # Create Prompt
    # =====================================

    prompt = f"""
    Answer the question based on the context below.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    # =====================================
    # Generate Response
    # =====================================

    result = generator(
      prompt,
      max_new_tokens=100,
      temperature=0.3,
      do_sample=True
    )

    answer = result[0]["generated_text"]
    answer = answer.replace(prompt, "").strip()

    # =====================================
    # Display Answer
    # =====================================

    print("\n🤖 AI Assistant:\n")

    print(answer)

    print("\n" + "="*60 + "\n")


🤖 PDF Chatbot Ready!
Type 'exit' to quit.

🧑 You: what the pdf was about

🤖 AI Assistant:

The pdf was about the history of artificial intelligence, including the first attempts at creating machines that could perform intelligent tasks, the first disappointment in the field, and the ALPAC report on machine translation.


🧑 You: what are the fundamentals of artificial intelligence

🤖 AI Assistant:

1. Artificial Intelligence is the ability of a computer or a robot controlled by a computer to perform various tasks in a manner similar to intelligent beings.
    2. Narrow AI is effective only in the specific areas it has been trained for, unlike human intelligence; it deals with a specific system and a specific problem.
    3. For example, fraud detection, facial recognition, and natural language processing.
    4. Artificial Intelligence


🧑 You: what is facial recognition

🤖 AI Assistant:

Facial recognition is a technology that uses algorithms to identify individuals based on their fac